# Reproduce F1 vs. instance count / signer count correlation figures

Regenerates `corr_mat_MViTv2_S.pdf` (F1 vs. training instance count) and
`corr_signers_MViTv2_S.pdf` (F1 vs. training signer count) for a given
WLASL split and model, using:

1. A results JSON with `cls_rep`, `all_targets`, `all_preds` keys
   (in this case `src/runs/asl2000/MViTv2_S/exp002/results/cls_rep_all_targets_preds.json`)
2. Per-class training-set stats (instance counts, signer distributions),
   loaded the same way as in your original stats notebook.

**Assumptions to verify** (see markdown notes inline below):
- `cls_rep` keys map to glosses via `get_class_list()` ordering when numeric.
- Instance/signer counts are drawn from the **training** set, not test/val.
- Signer count = number of unique signer IDs (dict keys), not total occurrences.
- Point shading = number of classes sharing an exact (x, F1) coordinate.

In [1]:
import json
from pathlib import Path

import pandas as pd

# locals -- same package used in your original stats notebook
from src.configs import get_class_list
from src.preprocess import WLASLClass
from src.run_types import RUNS_PATH
from src.stats import (
    AVAIL_SETS,
    AVAIL_SPLITS,
    get_per_instance_stats,
    reverse_preproc_format,
)
from src.video_dataset import (
    get_labels_path,
    get_wlasl_info,
    load_data_from_json,
)
from src.visualise2 import plot_metric_correlation, save_fig, set_thesis_style

set_thesis_style()  # shared thesis figure styling -- see src/VISUALISE2_CONVENTIONS.md

## Parameters

Adjust `split_name`, `model_name`, and `results_path` for whichever run you want to plot.

In [ ]:
verbosity = 1
def printv(*args, level=1, **kwargs):
    if level <= verbosity:
        print(*args, **kwargs)

split_name: AVAIL_SPLITS = "asl2000"  # change for a different split
model_name = "MViTv2_S"
exp_name = "exp002"  # change if your run used a different experiment id

results_path = RUNS_PATH / f"{split_name}/{model_name}/{exp_name}/results/cls_rep_all_targets_preds.json"


output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

classes = get_class_list()
printv(f"Num classes: {len(classes)}")

Num classes: 2000


## Load classification results and extract per-class F1

We read per-class F1 straight from `cls_rep`, since that's presumably the
same dict used to report your accuracy figures elsewhere in the paper --
safer than recomputing from `all_targets`/`all_preds`, which risks a
label-ordering mismatch if you don't have the exact label encoding used
at training/eval time.

In [3]:
with open(results_path) as f:
    results = {k : v for k, v in json.load(f).items()}
    
# with open(tp_path, 'r') as f:
#     tp_d = json.load(f)
    
# with open(cls_rep_path, 'r') as f:
#     cls_rep_d = json.load(f)
    
# results = {k : v for k, v in tp_d.items()} | {k : v for k, v in cls_rep_d.items()}

print(results.keys())

FileNotFoundError: [Errno 2] No such file or directory: '/home/luke/Code/SLR/src/runs/asl2000/MViTv2_S/exp002/results/cls_rep_all_targets_preds.json'

In [ ]:


cls_rep = results["cls_report"]
all_targets = results["all_targets"]
all_preds = results["all_preds"]

exclude_keys = {"accuracy", "macro avg", "weighted avg"}

per_class_rows = []
for key, metrics_dict in cls_rep.items():
    if key in exclude_keys:
        continue
    # cls_rep keys are either stringified numeric labels (map via classes list)
    # or already gloss strings -- handle both
    if key.isdigit():
        gloss = classes[int(key)]
    else:
        gloss = key
    per_class_rows.append({"gloss": gloss, "f1": metrics_dict["f1-score"]})

f1_df = pd.DataFrame(per_class_rows)
printv(f"Loaded F1 scores for {len(f1_df)} classes")
f1_df.head()

Loaded F1 scores for 2000 classes


,gloss,f1
0,book,0.600000
1,drink,0.666667
2,computer,0.750000
3,before,0.000000
4,chair,0.666667


## Load per-class training-set instance and signer counts

Uses the same loading path as your original stats notebook, but only for
the `train` set, since the original figures plot F1 (test set) against
*training* supervision available per class.

In [ ]:
set_name: AVAIL_SETS = "train"
# set_name: AVAIL_SETS = "test"

set_path_info = get_wlasl_info(split_name, set_name)
set_path = get_labels_path(set_name, set_path_info["labels"], set_path_info["label_suff"])
train_data = reverse_preproc_format(load_data_from_json(set_path, policy="strict"), classes)

train_stats = get_per_instance_stats([WLASLClass.model_validate(i) for i in train_data])

stats_rows = []
for gloss, stat in train_stats.items():
    num_instances = stat["num_instances"]
    signers_distribution = stat["signers_distribution"]  # {signer_id: occurrence_count}
    num_signers = len(signers_distribution)  # unique signer count
    stats_rows.append({"gloss": gloss, "num_instances": num_instances, "num_signers": num_signers})

stats_df = pd.DataFrame(stats_rows)
printv(f"Loaded stats for {len(stats_df)} classes")
stats_df.head()

Loaded stats for 2000 classes


,gloss,num_instances,num_signers
0,book,30,14
1,drink,25,15
2,computer,20,12
3,before,18,13
4,chair,19,11


In [ ]:
merged = f1_df.merge(stats_df, on="gloss", how="inner")

if len(merged) != len(f1_df):
    missing = set(f1_df["gloss"]) - set(merged["gloss"])
    print(f"WARNING: {len(missing)} glosses from results not found in training stats: {missing}")

merged.head()

,gloss,f1,num_instances,num_signers
0,book,0.600000,30,14
1,drink,0.666667,25,15
2,computer,0.750000,20,12
3,before,0.000000,18,13
4,chair,0.666667,19,11


## Plotting

Uses the shared `plot_metric_correlation` from `src/visualise2.py` (same convention as the rest
of the thesis figures) instead of a notebook-local function. Point shading reflects how many
classes share the exact same `(x, F1)` coordinate (`shade_by_density=True`, the default).

## Figure 1: F1 vs. instance count

In [ ]:
instance_plot_path = output_dir / f"corr_mat_{model_name}.pdf"
fig1, ax1, tau_inst, p_inst = plot_metric_correlation(
    merged["num_instances"],
    merged["f1"],
    title="Per-gloss instance count vs. recognition F1 score",
    xlabel="Number of instances",
)
save_fig(fig1, instance_plot_path)
printv(f"Saved to {instance_plot_path}")
print(f"Instance count correlation: tau={tau_inst:.3f}, p={p_inst:.3e}")

## Figure 2: F1 vs. signer count

In [ ]:
signers_plot_path = output_dir / f"corr_signers_{model_name}.pdf"
fig2, ax2, tau_signers, p_signers = plot_metric_correlation(
    merged["num_signers"],
    merged["f1"],
    title="Per-gloss signer count vs. recognition F1 score",
    xlabel="Number of unique signers",
)
save_fig(fig2, signers_plot_path)
printv(f"Saved to {signers_plot_path}")
print(f"Signer count correlation: tau={tau_signers:.3f}, p={p_signers:.3e}")